<a href="https://colab.research.google.com/github/Apham130/certiport-java-1/blob/main/Mosaic_Plots_for_disaggregated_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
### Run this cell.  Scroll Down and folow the instructions for Pasting Data.


import IPython.display as display
import ipywidgets as widgets
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from io import StringIO

# --- Global Widgets (to persist state) ---
if 'data_input_widget' not in globals():
    data_input_widget = widgets.Textarea(
        value='',
        placeholder='Paste your comma-separated data here (e.g., Gender,Treatment,Outcome\nMale,Medication,No Heart Attack)',
        description='Pasted Data:',
        disabled=False,
        layout={'height': '200px', 'width': '800px'}
    )

if 'process_button' not in globals():
    process_button = widgets.Button(
        description='Process Data and Identify Categories',
        button_style='success',
        tooltip='Click to process the pasted data and identify categories'
    )

if 'clear_button' not in globals():
    clear_button = widgets.Button(
        description='Clear Data',
        button_style='info',
        tooltip='Click to clear the pasted data and output'
    )

# New widgets for y-order input (declared globally to manage state)
if 'y_order_input_widget' not in globals():
    y_order_input_widget = widgets.Textarea(
        value='',
        placeholder='Enter y-variable categories in desired order, comma-separated (e.g., Cat1,Cat2,Cat3)',
        description='Y-Axis Order:',
        disabled=True, # Disabled until data is processed
        layout={'height': '100px', 'width': '800px'}
    )

if 'update_order_button' not in globals():
    update_order_button = widgets.Button(
        description='Generate Plot with Custom Order',
        button_style='warning',
        tooltip='Click to re-generate the plot with the specified y-axis order',
        disabled=True # Disabled until data is processed
    )

# Global variables to store processed data and metadata
_processed_df = None
_group_col_name = None
_x_col_name = None
_y_col_name = None
_x_levels = None
_y_levels_original = None # To store the original detected y_levels
_groups_levels = None

# --- Plotting Function (now separated) ---
def _plot_mosaic(df_raw, group_col_name, x_col_name, y_col_name, x_levels, y_levels_to_use, groups_levels):
    # Create a copy to avoid modifying the original _processed_df
    df = df_raw.copy()

    # Dynamic adjustment for x-axis labels based on the number of x_levels
    current_x_label_rotation = 30
    current_x_label_fontsize = 16
    if len(x_levels) > 5:
        current_x_label_rotation = 90
        current_x_label_fontsize = 10
    if len(x_levels) > 10:
        current_x_label_fontsize = 8 # Even smaller for very many categories

    # Convert y_col to categorical with specified order
    y_cat_type = pd.CategoricalDtype(categories=y_levels_to_use, ordered=True)
    df[y_col_name] = df[y_col_name].astype(y_cat_type)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 7), sharey=False)
    colors = plt.get_cmap("tab10").colors # Define colors once

    # --- LEFT PLOT: Population Mosaic (ignoring group_col) ---
    ax1.set_title("Population", fontsize=24, fontweight='bold', y=1.05) # Using y instead of pad
    ax1.set_ylabel(f"Relative Frequency of {y_col_name}", fontsize=14) # Reduced y-axis title font size
    ax1.set_xticks([])
    ax1.set_xlim(0,1)
    ax1.set_ylim(0,1)
    ax1.set_yticks([0, 0.2, 0.4, 0.6, 0.8, 1.0])
    ax1.set_yticklabels(['0%', '20%', '40%', '60%', '80%', '100%'], fontsize=12)

    total_population_count = len(df)
    current_x_pop = 0

    for x_val in x_levels:
        x_subset_pop = df[df[x_col_name] == x_val]
        x_count_pop = len(x_subset_pop)

        if x_count_pop == 0:
            continue

        width_pop = x_count_pop / total_population_count

        current_y_pop = 0
        # Get value counts for y_col in the specified order
        y_counts_pop = x_subset_pop[y_col_name].value_counts(normalize=False).reindex(y_levels_to_use, fill_value=0)

        for yi, y_val in enumerate(y_levels_to_use):
            cell_count_pop = y_counts_pop[y_val]
            if cell_count_pop == 0:
                continue
            height_pop = cell_count_pop / x_count_pop

            rect_pop = Rectangle(
                (current_x_pop, current_y_pop),
                width_pop,
                height_pop,
                facecolor=colors[yi % len(colors)],
                edgecolor="white",
                linewidth=2
            )
            ax1.add_patch(rect_pop)
            current_y_pop += height_pop

        ax1.text(
            current_x_pop + width_pop/2,
            -0.05,
            str(x_val),
            ha="right",
            va="top",
            rotation=current_x_label_rotation, # Using dynamically adjusted rotation
            fontsize=current_x_label_fontsize # Using dynamically adjusted font size
        )
        current_x_pop += width_pop

    # --- RIGHT PLOT: Grouped Mosaic with Spacing ---
    ax2.set_title(f"Grouped by {group_col_name}", fontsize=24, fontweight='bold', y=1.05) # Using y instead of pad
    ax2.set_ylabel('')
    ax2.set_xticks([])
    ax2.set_xlim(0,1)
    ax2.set_ylim(0,1)
    ax2.set_yticks([0, 0.2, 0.4, 0.6, 0.8, 1.0])
    ax2.set_yticklabels(['0%', '20%', '40%', '60%', '80%', '100%'], fontsize=12)

    total_data_points = len(df)
    group_relative_widths = {group: len(df[df[group_col_name] == group]) / total_data_points for group in groups_levels}

    spacing_factor = 0.01 # Half of previous 0.02, as requested (absolute width of gap)

    current_overall_x = 0
    num_groups = len(groups_levels)
    if num_groups > 1:
        effective_bar_width_total = 1.0 - (num_groups - 1) * spacing_factor
    else:
        effective_bar_width_total = 1.0

    for gi, group in enumerate(groups_levels):
        subset = df[df[group_col_name] == group]
        total_group_count = len(subset)

        if gi > 0:
            current_overall_x += spacing_factor

        group_x_axis_width = group_relative_widths[group] * effective_bar_width_total

        current_x_group = current_overall_x

        for x_val in x_levels:
            x_subset = subset[subset[x_col_name] == x_val]
            x_count = len(x_subset)

            if x_count == 0:
                continue

            width_group = (x_count / total_group_count) * group_x_axis_width

            current_y_group = 0
            # Get value counts for y_col in the specified order
            y_counts_group = x_subset[y_col_name].value_counts(normalize=False).reindex(y_levels_to_use, fill_value=0)

            for yi, y_val in enumerate(y_levels_to_use):
                cell_count = y_counts_group[y_val]
                if cell_count == 0:
                    continue
                height_group = cell_count / x_count

                rect = Rectangle(
                    (current_x_group, current_y_group),
                    width_group,
                    height_group,
                    facecolor=colors[yi % len(colors)],
                    edgecolor="white",
                    linewidth=1
                )
                ax2.add_patch(rect)
                current_y_group += height_group

            ax2.text(
                current_x_group + width_group/2,
                -0.05,
                str(x_val),
                ha="right",
                va="top",
                rotation=current_x_label_rotation, # Using dynamically adjusted rotation
                fontsize=current_x_label_fontsize # Using dynamically adjusted font size
            )
            current_x_group += width_group

        ax2.text(
            current_overall_x + group_x_axis_width/2,
            1.02,
            str(group),
            ha="center",
            va="bottom",
            fontsize=16,
            fontweight="bold"
        )

        current_overall_x += group_x_axis_width

    # Create a single legend for the figure
    handles_for_legend = [Rectangle((0,0),1,1, facecolor=colors[yi % len(colors)]) for yi, y_val in enumerate(y_levels_to_use)]
    fig.legend(
        handles_for_legend,
        y_levels_to_use,
        title=y_col_name,
        loc='upper left', # Changed to upper left
        bbox_to_anchor=(0.0, 1.0), # Placing legend at the very top of the figure
        bbox_transform=fig.transFigure,
        fontsize=10, # Reduced font size
        title_fontsize=12, # Reduced title font size
        frameon=True
    )

    # Removed plt.tight_layout() to avoid potential interference
    plt.subplots_adjust(top=0.8, bottom=0.1) # Significantly increased top margin to make ample space for legend and titles
    plt.show()

# --- Event Handlers ---
def process_data_and_display_options(b):
    global _processed_df, _group_col_name, _x_col_name, _y_col_name, _x_levels, _y_levels_original, _groups_levels

    with output_area:
        display.clear_output(wait=True)
        pasted_data = data_input_widget.value

        if not pasted_data.strip() or pasted_data.strip() == data_input_widget.placeholder:
            print("No data was pasted or default placeholder text remains. Please paste your data into the widget above.")
            return

        try:
            df_raw = pd.read_csv(
                StringIO(pasted_data.strip()),
                sep=","
            )

            if df_raw.shape[1] < 3:
                print("Error: Data must have at least 3 columns for Group, X, and Y variables.")
                return

            # Store metadata globally
            _group_col_name = df_raw.columns[0]
            _x_col_name = df_raw.columns[1]
            _y_col_name = df_raw.columns[2]

            # Explicitly strip whitespace from the y-column in the DataFrame
            _processed_df = df_raw[[_group_col_name, _x_col_name, _y_col_name]].dropna()
            _processed_df[_y_col_name] = _processed_df[_y_col_name].astype(str).str.strip()

            if _processed_df.empty:
                print("Error: After dropping missing values, no data remains for plotting.")
                return

            _x_levels = sorted(list(_processed_df[_x_col_name].unique()))
            # Now _processed_df[_y_col_name].unique() should already be stripped
            _y_levels_original = sorted(list(_processed_df[_y_col_name].unique()))
            _groups_levels = sorted(list(_processed_df[_group_col_name].unique()))

            print(f"Detected Y-variable categories: {_y_levels_original}")
            print("Please enter the desired order for these categories (bottom to top on the y-axis), comma-separated:")

            # Enable and pre-fill the y_order_input_widget
            y_order_input_widget.value = ','.join(_y_levels_original)
            y_order_input_widget.disabled = False
            update_order_button.disabled = False

            # Display the order input widgets AFTER processing
            display.display(y_order_input_widget, update_order_button)
            print("Y-axis categories detected. Adjust the 'Y-Axis Order' text area if needed, then click 'Generate Plot with Custom Order'.")

        except Exception as e:
            print(f"An error occurred while processing data: {e}")
            print("Please ensure your data is comma-separated and has at least 3 columns.")

def generate_plot_with_custom_order(b):
    global _processed_df, _group_col_name, _x_col_name, _y_col_name, _x_levels, _y_levels_original, _groups_levels

    with output_area:
        display.clear_output(wait=True)
        if _processed_df is None:
            print("Please process data first by clicking 'Process Data and Identify Categories'.")
            return

        custom_order_str = y_order_input_widget.value.strip()
        if not custom_order_str:
            print("No custom order entered. Using default alphabetical order.")
            y_levels_to_use = _y_levels_original
        else:
            y_levels_parsed = [item.strip() for item in custom_order_str.split(',')]
            # Validate custom order against original detected levels
            # Check if all elements in parsed are in original and vice-versa
            if not (set(y_levels_parsed).issubset(set(_y_levels_original)) and set(_y_levels_original).issubset(set(y_levels_parsed))):
                print(f"Error: Custom order contains invalid or missing categories.\nOriginal: {_y_levels_original}\nProvided: {y_levels_parsed}")
                print("Please ensure all categories are present and correctly spelled.")
                y_levels_to_use = _y_levels_original # Fallback to original
            else:
                y_levels_to_use = y_levels_parsed

        print(f"Generating plot with Y-axis order: {y_levels_to_use}")
        _plot_mosaic(_processed_df, _group_col_name, _x_col_name, _y_col_name, _x_levels, y_levels_to_use, _groups_levels)

def clear_data_and_output(b):
    global _processed_df, _group_col_name, _x_col_name, _y_col_name, _x_levels, _y_levels_original, _groups_levels

    data_input_widget.value = ''  # Clear the text area
    y_order_input_widget.value = '' # Clear order input
    y_order_input_widget.disabled = True
    update_order_button.disabled = True

    # Reset global stored data
    _processed_df = None
    _group_col_name = None
    _x_col_name = None
    _y_col_name = None
    _x_levels = None
    _y_levels_original = None
    _groups_levels = None

    with output_area:
        display.clear_output(wait=True)
        print("Data cleared. Please paste new comma-separated data into the Textarea above, then click 'Process Data and Identify Categories'.")

# --- Link Buttons to Functions ---
process_button.on_click(process_data_and_display_options)
clear_button.on_click(clear_data_and_output)
update_order_button.on_click(generate_plot_with_custom_order)

# Create an output widget to display plots and messages
if 'output_area' not in globals():
    output_area = widgets.Output()

# --- Display Initial Widgets ---
display.display(data_input_widget, widgets.HBox([process_button, clear_button]), output_area)

# On first run, display initial message inside the output_area
with output_area:
    if not data_input_widget.value.strip():
        print("Please paste your comma-separated data into the Textarea above, then click 'Process Data and Identify Categories'.")
    else:
        # If data is already there (e.g., after previous run/refresh), try to process it automatically
        process_data_and_display_options(None)


Textarea(value='Dept,Gender,Status\nC,Male,Admitted\nC,Female,Rejected\nD,Male,Rejected\nF,Male,Rejected\nA,Ma…

Output()